# 11 · A different pattern: encounter with a relative 🧬

![The rainbow Beast (left) beside its Turing-patterned relative (right)](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/turing.jpg)

How does a featureless ball of cells — an early embryo — decide *where* to put spots, stripes, fingers? 

Turing's model: A **reaction–diffusion** PDE system: 
* Two substances, an **activator** and an **inhibitor**,
* both react with each other
* and diffuse at **different speeds**;

That imbalance can make a perfectly uniform state spontaneously break up into a regular pattern. 

The same mechanism is thought to paint **leopard spots, zebra stripes, seashells and fish** —
*morphogenesis*, pattern from no pattern.

We put Turing's mechanism on the **surface of the sculpture itself**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from netgen.occ import OCCGeometry, Sphere, Cylinder, Glue, Pnt, X, Y, Z
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
import sys

In [ ]:
def progress(i, n):                                    # a tiny dependency-free bar (live only)
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:  # (survives JupyterLite/Colab/local)
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  growing the coat… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The beast's skin — a surface mesh

The beast is the **solid sculpture** of notebook 2 (a spherical shell bored by three
cylinders). We want a mesh of its **surface only** — a 2-D manifold in 3-D. **Two routes:**

1. **Faces → mesh** (used here): keep the solid's `faces`, `Glue` them into a closed
   **shell**, and mesh *that*.
2. **Solid → stop at the surface**: mesh the solid but halt after the surface step with
   `perfstepsend=MeshingStep.MESHSURFACE` — no volume mesh is ever built.

Both yield the same surface mesh. On it the measure is **`ds`** (surface area). We keep the
mesh **coarse** and resolve the pattern by **high order** ($k=4$) instead of tiny triangles —
fewer, bigger curved elements, each carrying a quartic field.

In [ ]:
def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)     # centre + shrink

solid = beast_sculpture()

# Route 1 — glue the faces into a closed shell, mesh the surface:
shell = Glue([f for f in solid.faces])
mesh = Mesh(OCCGeometry(shell).GenerateMesh(maxh=0.75))          # coarse — high order resolves the pattern
mesh.Curve(3)
Draw(mesh)

# Route 2 (alternative) — mesh the solid but STOP after surface meshing (no volume elements):
#mesh_alt = Mesh(OCCGeometry(solid).GenerateMesh(maxh=0.75, perfstepsend=MeshingStep.MESHSURFACE))
#mesh_alt.Curve(3)
#Draw(mesh_alt)

## 2. The model — an activator and an inhibitor

The **Gray–Scott** system for two surface concentrations $u$ (substrate) and $v$ (activator):
$$ \partial_t u = D_u\,\Delta_\Gamma u - u v^2 + F(1-u),\qquad
   \partial_t v = D_v\,\Delta_\Gamma v + u v^2 - (F+k)\,v . $$

- $v$ is **autocatalytic**: $uv^2$ converts substrate $u$ into *more* $v$.
- $F$ **feeds** fresh $u$; $k$ **removes** $v$.
- The activator diffuses **slower** than the substrate ($D_v<D_u$) — *short-range
  activation, long-range inhibition*, exactly Turing's recipe.
- Tune $F,k$ for **dots ↔ stripes ↔ coral**; $\Delta_\Gamma$ is the **surface** Laplacian.

The whole pipeline at a glance:

![Gray-Scott pipeline: surface, equations, seeding, time stepping, emerging pattern](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/gray-scott-model.webp)

> **Why `grad(u).Trace()`?**
>
> On a surface mesh the elements are **boundary** elements, so a plain `grad(u)` is invalid
> inside a `ds`-form — take its **tangential trace** `grad(u).Trace()` (the surface gradient
> $\nabla_\Gamma u$). Without it, assembly aborts: *"Trialfunction does not support BND-forms,
> maybe a Trace() operator is [missing]"*.

In [ ]:
fes = H1(mesh, order=4)            # high order k=4 on the coarse mesh (p-refinement)
u, w = fes.TnT()
M = BilinearForm(u * w * ds, check_unused=False).Assemble()                    # surface mass
K = BilinearForm(grad(u).Trace() * grad(w).Trace() * ds, check_unused=False).Assemble()  # surface stiffness

Du, Dv, F, k = 3.6e-3, 1.8e-3, 0.037, 0.060        # "coral" regime; note D_v = D_u/2
dt = 1.0

## 3. A variational semi-implicit (IMEX) scheme

We take diffusion **implicitly**; the local **reaction** stays **explicit**. 

Each species then needs one pre-factorised solve per step, $M+\Delta t\,D\,K$ (s.p.d.). 
The reaction is a **nonlinear form** we never assemble: we just **`Apply`** it
to the current state each step.

In [ ]:
def factor(D):
    mstar = M.mat.CreateMatrix()
    mstar.AsVector().data = M.mat.AsVector() + dt * D * K.mat.AsVector()
    return mstar.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

inv_u, inv_v = factor(Du), factor(Dv)

gfu, gfv = GridFunction(fes), GridFunction(fes)
react_u = BilinearForm(fes, nonassemble=True)
react_u += -(-u * gfv * gfv + F * (1 - u)) * w * ds
react_v = BilinearForm(fes, nonassemble=True)
react_v += -(gfu * u * u - (F + k) * u) * w * ds

## 4. Seed the beast and let the pattern grow

We start from a calm skin ($u=1$, $v=0$) and dab a few **patches** of activator onto
the surface. 

In [ ]:
# A few smooth activator dabs. On a surface mesh the elements ARE boundary elements, so the
# fields are set with **definedon=mesh.Boundaries(".*")** (an L2-projection onto the surface).
seed = sum(exp(-((x - px)**2 + (y - py)**2 + (z - pz)**2) / 2.0)
           for (px, py, pz) in [(4, 0, 0), (0, 4, 0), (0, 0, 4),
                                (-4, 0, 0), (0, -4, 0), (0, 0, -4)])
gfu.Set(1 - 0.5 * seed, definedon=mesh.Boundaries(".*"))    # substrate u
gfv.Set(0.25 * seed,    definedon=mesh.Boundaries(".*"))    # activator v

no_grid = {"Objects": {"Wireframe": False, "Edges": False},
           "Multidim": {"speed": 0.3, "animate": True}}     # play the growth slowly

Draw(gfv, mesh, "u/v", order=3, min=0, max=1, autoscale=False, settings=no_grid)

In [ ]:
res = gfu.vec.CreateVector()
nsteps = 6500
nsamples = 4
growth = GridFunction(fes, multidim=0)                      # activator v snapshots, at i * nsteps / 4
growth_u = GridFunction(fes, multidim=0)                    # substrate u snapshots, same times
growth.AddMultiDimComponent(gfv.vec); growth_u.AddMultiDimComponent(gfu.vec)   # i = 0: the bare seed
snap_at = {round(i * nsteps / nsamples) for i in range(1,nsamples)}        # i = 1, 2, 3 (mind the rounding)
with TaskManager():
    for step in range(nsteps):
        react_u.Apply(gfu.vec, res); gfu.vec.data = inv_u * (M.mat * gfu.vec - dt * res).Evaluate()
        react_v.Apply(gfv.vec, res); gfv.vec.data = inv_v * (M.mat * gfv.vec - dt * res).Evaluate()
        if step + 1 in snap_at:
            growth.AddMultiDimComponent(gfv.vec); growth_u.AddMultiDimComponent(gfu.vec)
        progress(step, nsteps)

Time evolution (with 4 samples; you may increase nsamples):

In [ ]:
Draw(growth, mesh, "v", order=3, min=0, max=0.3, autoscale=False,
     interpolate_multidim=True, animate=True, settings=no_grid)

The **substrate** $u$ tells the inverse story — it is **consumed** wherever the activator
blooms, so its labyrinth is the photographic negative of the coat above.

In [ ]:
Draw(growth_u, mesh, "u", order=3, min=0, max=1, autoscale=False,
     interpolate_multidim=True, animate=True, settings=no_grid)

Pre-rendered results:
![Two Beasts side by side — the activator's labyrinth in warm browns and the substrate's inverse pattern in cool teal, both spinning](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/turing.gif)

## Supplementary — VTK output for ParaView, PyVista & Blender

That offline render went through **PyVista**. The bridge from NGSolve to *any* external tool
(ParaView, PyVista, Blender) is a **VTK** file — the mesh plus one or more coefficient functions,
written in a single call:

```python
vtk = VTKOutput(mesh, coefs=[gfv, gfu], names=["activator", "substrate"],
                filename="turing", subdivision=2)
vtk.Do()                  # -> turing.vtu ; open in ParaView, or pyvista.read("turing.vtu")
```

A **high-order** field must be sampled into the (cell-wise) VTK file — two knobs:

- **`subdivision=N`** — split each element into $2^N$ pieces and sample the field **linearly** at
  the sub-vertices: more, smaller **linear** cells, readable by *every* VTK tool.
- **`order=M`** *(new)* — write native **higher-order** (Lagrange) VTK cells of order $M$: far
  fewer cells, exact to order $M$ — but needs a recent reader (ParaView ≥ 5.5).

Use `subdivision` for portability, `order` for compact high-order fidelity. Then **ParaView** to
explore, **PyVista** for scripted renders (as in `render_turing_video.py`), and **Blender** (via
VTK / `.x3d` import) for cinematic shading.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("10-nonlinear-allencahn", "10 · Nonlinear problems — Allen–Cahn & Newton")
    _next = ("12-pedestrian-dynamics", "12 · A crowd heads for coffee 🚶☕")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))